In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, initcap

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"
# Read constructors data from bronze layer
bronze_constructors_df = spark.table(bronze_table)

In [0]:
# Apply transformations
silver_constructors_df = (
    bronze_constructors_df
    # Remove records with NULL constructor_id
    .filter(col("constructorId").isNotNull())
    # Remove duplicates
    .dropDuplicates(["constructorId"])
    # Drop URL column
    .drop("url")
    # Rename columns to snake_case and meaningful names
    .withColumnsRenamed(
        {
            "constructorId": "constructor_id",
            "name": "constructor_name"
        }
    )
    # Transform nationality to Title Case
    .withColumn("nationality", initcap(col("nationality")))
)

# print(f"Total records after transformations: {silver_constructors_df.count()}")
# display(silver_constructors_df)

In [0]:
# Write transformed data to silver layer
(silver_constructors_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table))

print(f"✓ Constructors data successfully written to {catalog_name}.{silver_schema}.constructors")

In [0]:
# spark.table(silver_table).limit(10).display()